# GDL - Midterm n.1
**Student ID: 727009** 
**Submission ID: 053** 

In the first midterm assignment, you are required to implement a Gaussian Mixture Model (GMM) and the Expectation-Maximization (EM) algorithm. You are allowed to use `numpy` and other non-machine learning libraries, e.g., `pandas`, `matplotlib`.

**Assumptions.** To ease the implementation, we assume, for each Gaussian distribution in the mixture $\mathcal{N}(\mu_k, \Sigma_k)$, that the *covariance matrix is diagonal*. Furthermore, keep in mind that a good implementation of the EM algorithm provides monotonically increasing log-likelihood, *but* can get stuck in local minima; initialization strategies are fundamental (random? sample points? k-means++?).

**Hyperparameter $k$.** When using a GMM, you should ask yourself: *how many mixtures will be in the data?* You can *maximize* the Bayesian Information Criterion (BIC) to select the number of categories $k$ on your training set. Formally,
$$
\text{BIC}
=
\log P(X\mid \theta) - \frac{|\theta|}{2}\log n,
$$
where $n$ is the number of samples in the training set, $\theta=\{\pi,\mu,\sigma\}$ the parameters of the GMM and $|\theta|$ the number of parameters, i.e., the sum of the number of parameters in $\pi$, $\mu$, and $\sigma$ (*hint*: it also depends on the dimensionality of data $d$!).

**Summary.** Overall, you are required to:
1.  **Implement the GMM class**. Fill the `log_likelihood(samples)`.
2. **Implement the EM algorithm**. Fill the `fit(samples)` method to fit the parameters of a GMM.
3. **Implement the BIC score**. Fill the `bic(samples)` method to perform model selection.
4. **Run training and evaluation**. Select the best scoring model using BIC (Bayesian Information Criterion) for values of $k=\{1,\ldots,6\}$.

**Evaluation.** Your solution will be tested against an hidden test set. For your learning experience, we require you to refrain from using LLM generated code. Violations will be flagged and invalidate the midterm. **Do not alter Sections 3 and 4 of the notebook.**

### 1. Libraries

### 1. Libraries

In [11]:
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import logsumexp
import os
os.chdir(r'C:\Users\iasmi\Documents\GitHub\GenerativeAndDeepLearning\Midterms\Midterm1')

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)

print("Libraries imported successfully.")

Libraries imported successfully.


### 2. Gaussian Mixture Model (GMM)

Feel free to play around the implementation. **For evaluation purposes**, the class **must** expose the following attributes and methods:
- `_pi: np.ndarray`
- `_mu: np.ndarray`
- `_sigma: np.ndarray`
- `fit(samples: np.ndarray)`
- `bic(samples: np.ndarray) -> np.ndarray`
- `log_likelihood(samples: np.ndarray) -> np.ndarray`

In [12]:
class GaussianMixtureModel:
    def __init__(self, n_categories: int, n_features: int):
        self.n_categories = n_categories


        
        #init 
        self.n_features = n_features

        self._pi = np.full(n_categories, 1.0 / n_categories) #init with equal weights

        self._mu = None  #initialized with data later when fit() sees the data

        self._sigma = np.ones((n_categories, n_features)) #storing one variance for each feature


    def _kmeans_plus_plus(self, samples):
        #using kmeans strategy to initialize cloud centers
        n_samples = samples.shape[0]
        idx = np.random.randint(0, n_samples)
        centers = [samples[idx]]

        for _ in range(self.n_categories - 1):
            #for each sample, I'm computing the distant from the closest center
            dists = np.array([min(np.sum((x - c)**2) for c in centers) for x in samples])
            #chosing the next center with probability proportional to distance
            probs = dists / dists.sum()
            idx = np.random.choice(n_samples, p=probs)
            centers.append(samples[idx])
        return np.array(centers)

    def log_likelihood(self, samples: np.ndarray) -> np.ndarray:
        """Computes logP(X|θ)"""

        n_samples = samples.shape[0]

        #matrix to stores the log-probability of each sample under each component
        log_probs = np.zeros((n_samples, self.n_categories))

        for k in range(self.n_categories):

            #Gaussian formula in parts

            exponent = -0.5 * np.sum(((samples - self._mu[k])**2) / self._sigma[k], axis=1)

            normalization = -0.5 * (self.n_features * np.log(2 * np.pi) + np.sum(np.log(self._sigma[k])))
            
            #using logsumexp for numerical stability (easier to sum bigger numbers)
            log_probs[:, k] = np.log(self._pi[k] + 1e-10) + normalization + exponent

        total_log_likelihood = np.sum(logsumexp(log_probs, axis=1))
        
        return total_log_likelihood

    def fit(self, samples: np.ndarray, max_iter: int = 100, tol: float = 1e-4):
        """Fits the GMM parameters using the EM algorithm."""

        n_samples = samples.shape[0]
        #strategy or smart initialization
        self._mu = self._kmeans_plus_plus(samples)
        self._sigma = np.ones((self.n_categories, self.n_features))
        self._pi = np.full(self.n_categories, 1.0 / self.n_categories)
        #avoiding that the variance becomes zero
        self._sigma = np.maximum(self._sigma, 1e-6)
        prev_log_likelihood = -np.inf

        for iteration in range(max_iter):
            #E-step
            #computing responsibilities
            log_resp = np.zeros((n_samples, self.n_categories))
            for k in range(self.n_categories):
                exponent = -0.5 * np.sum(((samples - self._mu[k])**2) / self._sigma[k], axis=1)
                normalization = -0.5 * (self.n_features * np.log(2 * np.pi) + np.sum(np.log(self._sigma[k])))
                log_resp[:, k] = np.log(self._pi[k]) + normalization + exponent

            #normalizing in log-space, then going back to normal space
            log_denom = logsumexp(log_resp, axis=1, keepdims=True)
            gamma_ik = np.exp(log_resp - log_denom)

            #M-step
            #updating weights, means and diagonal variances

            Nk = np.sum(gamma_ik, axis=0)
            self._pi = Nk / n_samples
            self._mu = np.dot(gamma_ik.T, samples) / Nk[:, np.newaxis]
            for k in range(self.n_categories):
                diff = samples - self._mu[k]
                self._sigma[k] = np.sum(gamma_ik[:, k][:, np.newaxis] * (diff**2), axis=0) / Nk[k]
            #again, keeping it positive
            self._sigma = np.maximum(self._sigma, 1e-6)

            # convergência
            current_log_likelihood = self.log_likelihood(samples)
            if np.abs(current_log_likelihood - prev_log_likelihood) < tol:
                break
            prev_log_likelihood = current_log_likelihood
            


    def bic(self, samples: np.ndarray) -> np.ndarray:
        """Computes the BIC score."""

        n_samples = samples.shape[0]

        #computing the log-likelihood of the data under the model
        log_likelihood_value = self.log_likelihood(samples)
       
        #(k - 1) for mixture weights
        #k * d for means
        #k * d for diagonal variances
        num_parameters = (self.n_categories - 1) + \
                         (self.n_categories * self.n_features) + \
                         (self.n_categories * self.n_features)
        
        #BIC formula
        bic_score = log_likelihood_value - (num_parameters / 2) * np.log(n_samples)
        
        return bic_score
    

### 3. Training

Trains the model for increasing number of categories and selects the best scoring one in terms of BIC score.

In [13]:
# load training set
train_df = pd.read_csv('train.csv')
print(f"train.csv loaded successfully. n={train_df.shape[0]}, d={train_df.shape[1]}")

# model selection using bayesian information score
seed_everything(9951)
candidate_categories = range(1, 7)
max_bic, best_k, best_gmm = -np.inf, None, None
for k in candidate_categories:
    gmm = GaussianMixtureModel(n_categories=k, n_features=train_df.shape[1])
    gmm.fit(train_df.values)
    ll = gmm.log_likelihood(train_df.values)
    bic_score = gmm.bic(train_df.values)
    print(f"k={k}\tBIC={bic_score:.4f}\tlogP(X|θ)={ll:.4f}")
    if bic_score > max_bic:
        max_bic = bic_score
        best_k = k
        best_gmm = gmm

# print training info
print()
print("Best model")
print(f"k:\t\t{best_k}")
print(f"BIC:\t\t{max_bic:.4f}")
print(f"logP(X|θ):\t{best_gmm.log_likelihood(train_df.values):.4f}")

train.csv loaded successfully. n=800, d=5
k=1	BIC=-8320.1771	logP(X|θ)=-8286.7541
k=2	BIC=-8052.0416	logP(X|θ)=-7981.8532
k=3	BIC=-7919.3844	logP(X|θ)=-7812.4307
k=4	BIC=-7869.8688	logP(X|θ)=-7726.1497
k=5	BIC=-7899.2187	logP(X|θ)=-7718.7342
k=6	BIC=-7923.0582	logP(X|θ)=-7705.8083

Best model
k:		4
BIC:		-7869.8688
logP(X|θ):	-7726.1497


### 4. Evaluation

To check if you did not break the API, you can use the training file to run the tests.

This is not meaningful for the evaluation of your model, as the true test set is hidden.

In [14]:
# If test.csv does not exist copy train into test
!test -f test.csv || cp train.csv test.csv

'test' is not recognized as an internal or external command,
operable program or batch file.
'cp' is not recognized as an internal or external command,
operable program or batch file.


In [15]:
# load hidden test set ☠️
test_df = pd.read_csv('test.csv')
print(f"test.csv loaded successfully. n={test_df.shape[0]}, d={test_df.shape[1]}")

# print log-likelihood
test_log_likelihood = best_gmm.log_likelihood(test_df.values)
print(f"Log-likelihood of test data: {test_log_likelihood:.4f}")

# print parameters
print(f"k: {best_gmm.n_categories}")
print()
for cat_id in range(best_gmm.n_categories):
  print(f"π[{cat_id}]:", best_gmm._pi[cat_id])
  print(f"μ[{cat_id}]:", best_gmm._mu[cat_id])
  print(f"σ[{cat_id}]:", best_gmm._sigma[cat_id])
  print()

test.csv loaded successfully. n=800, d=5
Log-likelihood of test data: -7726.1497
k: 4

π[0]: 0.4351779221193148
μ[0]: [-2.01551759  0.5609366  -2.33861555 -0.50669088 -0.10953472]
σ[0]: [2.56338747 2.24906925 1.23115131 1.64866151 2.78841882]

π[1]: 0.12077368031814403
μ[1]: [-2.94819227 -1.75689689  0.81471515  2.46361187 -0.09986235]
σ[1]: [2.33105873 2.58037918 0.76326296 2.37020999 0.66202148]

π[2]: 0.3233475974254958
μ[2]: [-2.76492547 -0.57486467 -0.45676258  2.0508323  -3.81489372]
σ[2]: [1.56482996 3.15757033 1.05201721 2.20578264 1.55923917]

π[3]: 0.12070080013704584
μ[3]: [ 1.87924772  2.39716545 -0.1970316   0.26452232 -0.80132096]
σ[3]: [0.49252699 1.76528889 1.39096642 1.55278947 3.24692422]

